In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from sklearn.metrics import accuracy_score, fbeta_score, precision_score, recall_score

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.config import ARTIFACTS_DIR, DEFAULT_RUN_TAG, K2P_FILE, KOI_TEST_FILE, get_run_artifact_dirs
from exoplanet_detector.features.feature_selection import FINAL_FEATURE_COLUMNS
from exoplanet_detector.models.feature_analysis import build_evaluation_sets, load_deployed_models
from exoplanet_detector.visualization.plots import (
    plot_confusion_matrix_binary,
    plot_pr_curve_binary,
    plot_roc_curve_binary,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)

DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]
DEPLOY_MODELS_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_models.joblib"

VISUALIZATION_ARTIFACT_DIR = ARTIFACTS_DIR / "visualization" / RUN_TAG
VISUALIZATION_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_MANIFEST_PATH = VISUALIZATION_ARTIFACT_DIR / "plot_manifest.csv"

deployed_models, deployed_model_registry_df = load_deployed_models(DEPLOY_MODELS_PATH)

KOI_test_set = pd.read_csv(KOI_TEST_FILE)
K2P_set = pd.read_csv(K2P_FILE)

evaluation_sets = build_evaluation_sets(
    KOI_test_set,
    K2P_set,
    feature_columns=FINAL_FEATURE_COLUMNS,
    include_combined=False,
)

print(f"Visualization artifacts: {VISUALIZATION_ARTIFACT_DIR}")
print(f"Datasets: {list(evaluation_sets.keys())}")
deployed_model_registry_df


Visualization artifacts: /Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1
Datasets: ['KOI_test', 'K2P']


,deploy_id,model_name,profile,threshold
0,deploy_f2,knn,f2,0.363636
1,deploy_precision,hgb,precision_constrained,0.885432
2,deploy_recall,rf,recall_constrained,0.029039


In [3]:
def predict_with_tuned_threshold(deployed_spec, x_eval: pd.DataFrame):
    tuned_model = deployed_spec["model"]
    base_estimator = tuned_model.estimator_

    if not hasattr(base_estimator, "predict_proba"):
        raise ValueError("Diagnostics require estimators with predict_proba.")

    classes = np.asarray(tuned_model.classes_)
    if classes.shape[0] != 2:
        raise ValueError("Diagnostics expect binary classification models.")

    positive_label = 1 if 1 in classes else classes[-1]
    negative_label = [label for label in classes if label != positive_label][0]

    estimator_classes = np.asarray(base_estimator.classes_)
    pos_idx = int(np.where(estimator_classes == positive_label)[0][0])

    y_score = base_estimator.predict_proba(x_eval)[:, pos_idx]
    threshold = float(deployed_spec["threshold"])
    y_pred = np.where(y_score >= threshold, positive_label, negative_label)
    return y_pred, y_score, positive_label, negative_label, threshold


In [4]:
rows = []

for deploy_id, spec in deployed_models.items():
    model_name = spec["model_name"]
    profile = spec["profile"]

    for dataset_name, (x_eval, y_eval) in evaluation_sets.items():
        y_pred, y_score, pos_label, neg_label, threshold = predict_with_tuned_threshold(spec, x_eval)
        labels = (neg_label, pos_label)

        target_dir = VISUALIZATION_ARTIFACT_DIR / deploy_id / dataset_name
        target_dir.mkdir(parents=True, exist_ok=True)

        confusion_path = target_dir / "confusion_matrix.png"
        roc_path = target_dir / "roc_curve.png"
        pr_path = target_dir / "pr_curve.png"

        fig, _ = plot_confusion_matrix_binary(
            y_eval,
            y_pred,
            labels=labels,
            normalize="true",
            title=f"{deploy_id} | {dataset_name} | Confusion Matrix",
        )
        fig.savefig(confusion_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        fig, _, roc_auc = plot_roc_curve_binary(
            y_eval,
            y_score,
            title=f"{deploy_id} | {dataset_name} | ROC Curve",
        )
        fig.savefig(roc_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        fig, _, average_precision = plot_pr_curve_binary(
            y_eval,
            y_score,
            title=f"{deploy_id} | {dataset_name} | PR Curve",
        )
        fig.savefig(pr_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        rows.append(
            {
                "run_tag": RUN_TAG,
                "deploy_id": deploy_id,
                "model_name": model_name,
                "profile": profile,
                "dataset": dataset_name,
                "threshold": threshold,
                "positive_label": pos_label,
                "negative_label": neg_label,
                "accuracy": float(accuracy_score(y_eval, y_pred)),
                "recall": float(recall_score(y_eval, y_pred, zero_division=0)),
                "precision": float(precision_score(y_eval, y_pred, zero_division=0)),
                "f2": float(fbeta_score(y_eval, y_pred, beta=2, zero_division=0)),
                "roc_auc": roc_auc,
                "average_precision": average_precision,
                "confusion_matrix_path": str(confusion_path.resolve()),
                "roc_curve_path": str(roc_path.resolve()),
                "pr_curve_path": str(pr_path.resolve()),
            }
        )

plot_manifest_df = (
    pd.DataFrame(rows)
    .sort_values(["deploy_id", "dataset"], ascending=[True, True])
    .reset_index(drop=True)
)
plot_manifest_df.to_csv(PLOT_MANIFEST_PATH, index=False)

expected_plots = len(deployed_models) * len(evaluation_sets) * 3
print(f"Saved plot manifest: {PLOT_MANIFEST_PATH}")
print(f"Expected plots: {expected_plots}")
print(f"Generated plots: {plot_manifest_df.shape[0] * 3}")
plot_manifest_df


Saved plot manifest: /Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/plot_manifest.csv
Expected plots: 18
Generated plots: 18


,run_tag,deploy_id,model_name,profile,dataset,threshold,positive_label,negative_label,accuracy,recall,precision,f2,roc_auc,average_precision,confusion_matrix_path,roc_curve_path,pr_curve_path
0,v1,deploy_f2,knn,f2,K2P,0.363636,1,0,0.781928,0.908621,0.804580,0.885714,0.839045,0.909328,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/pr_curve.png
1,v1,deploy_f2,knn,f2,KOI_test,0.363636,1,0,0.887937,0.958106,0.781575,0.916696,0.965551,0.936924,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/pr_curve.png
2,v1,deploy_precision,hgb,precision_constrained,K2P,0.885432,1,0,0.469880,0.244828,0.986111,0.288149,0.883693,0.945650,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/pr_curve.png
3,v1,deploy_precision,hgb,precision_constrained,KOI_test,0.885432,1,0,0.819380,0.511840,0.979094,0.565848,0.976550,0.958278,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/pr_curve.png
4,v1,deploy_recall,rf,recall_constrained,K2P,0.029039,1,0,0.707229,0.998276,0.705238,0.921681,0.870669,0.945561,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/pr_curve.png
5,v1,deploy_recall,rf,recall_constrained,KOI_test,0.029039,1,0,0.680290,0.998179,0.531008,0.848823,0.969514,0.944598,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/pr_curve.png


In [5]:
plot_manifest_df[
    [
        "deploy_id",
        "dataset",
        "threshold",
        "f2",
        "roc_auc",
        "average_precision",
        "confusion_matrix_path",
        "roc_curve_path",
        "pr_curve_path",
    ]
]


,deploy_id,dataset,threshold,f2,roc_auc,average_precision,confusion_matrix_path,roc_curve_path,pr_curve_path
0,deploy_f2,K2P,0.363636,0.885714,0.839045,0.909328,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/K2P/pr_curve.png
1,deploy_f2,KOI_test,0.363636,0.916696,0.965551,0.936924,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_f2/KOI_test/pr_curve.png
2,deploy_precision,K2P,0.885432,0.288149,0.883693,0.945650,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/K2P/pr_curve.png
3,deploy_precision,KOI_test,0.885432,0.565848,0.976550,0.958278,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_precision/KOI_test/pr_curve.png
4,deploy_recall,K2P,0.029039,0.921681,0.870669,0.945561,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/K2P/pr_curve.png
5,deploy_recall,KOI_test,0.029039,0.848823,0.969514,0.944598,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/confusion_matrix.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/roc_curve.png,/Users/janma/Desktop/exoplanet-detection/artifacts/visualization/v1/deploy_recall/KOI_test/pr_curve.png
